# Fase 4 — Avaliação no Conjunto de Teste

Avaliação rigorosa dos dois modelos no conjunto de teste (nunca visto em treino/validação): mAP, IoU, precisão/recall, matriz de confusão, e análise de erros comentada. Requer os pesos treinados em `02_detection.ipynb` e `03_segmentation.ipynb`.

## Setup

In [ ]:
!pip install -q ultralytics opencv-python pandas pyarrow matplotlib pillow
# --upgrade e necessario: o Colab ja vem com uma versao antiga (2.0.2) do
# pacote kaggle pre-instalada, que nao suporta o token novo nem "python -m kaggle".
!pip install -q --upgrade kaggle

from pathlib import Path

# Armazenamento persistente compartilhado entre os notebooks: monta o Google
# Drive e usa uma pasta fixa. Troque o caminho se preferir outra estrutura.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/vc-seguranca-trabalho')
except ImportError:
    # Execucao fora do Colab (teste local) - usa uma pasta local.
    PROJECT_DIR = Path('./vc-seguranca-trabalho').resolve()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
ROOT = PROJECT_DIR
print("Diretorio do projeto:", ROOT)


## 11. Avaliação no conjunto de teste

`model.val(split="test")` para os dois modelos: mAP@0.5, mAP@0.5:0.95, precisão, recall, matriz de confusão, e IoU médio (detector).

In [ ]:
import json

from ultralytics import YOLO
from ultralytics.utils.metrics import box_iou
import torch

DETECTION_WEIGHTS = ROOT / "models" / "detection" / "css_yolov8n_baseline" / "weights" / "best.pt"
DETECTION_DATA_YAML = ROOT / "data" / "raw" / "construction-site-safety" / "data.yaml"
DETECTION_LABELS_DIR = ROOT / "data" / "raw" / "construction-site-safety" / "labels"
DETECTION_TEST_LIST = ROOT / "data" / "splits" / "construction-site-safety" / "test.txt"

SEGMENTATION_WEIGHTS = ROOT / "models" / "segmentation" / "coco_person_yolov8n_seg_baseline" / "weights" / "best.pt"
SEGMENTATION_DATA_YAML = ROOT / "data" / "raw" / "coco-person" / "data.yaml"

RESULTS_DIR = ROOT / "reports" / "test-evaluation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CONF_THRESHOLD = 0.25
IOU_MATCH_THRESHOLD = 0.5


def summarize_val_metrics(results):
    return {
        "precision": float(results.box.mp),
        "recall": float(results.box.mr),
        "mAP50": float(results.box.map50),
        "mAP50-95": float(results.box.map),
    }


def compute_mean_iou_detection():
    """Roda o detector no conjunto de teste, casa cada predicao com a caixa
    de mesma classe com maior IoU no ground-truth, e retorna o IoU medio dos
    pares casados (TP) com confianca >= CONF_THRESHOLD."""
    model = YOLO(str(DETECTION_WEIGHTS))
    test_images = [
        ROOT / "data" / "raw" / "construction-site-safety" / "images" / name
        for name in DETECTION_TEST_LIST.read_text(encoding="utf-8").splitlines()
        if name.strip()
    ]

    ious = []
    for img_path in test_images:
        result = model.predict(source=str(img_path), conf=CONF_THRESHOLD, verbose=False)[0]
        pred_boxes = result.boxes.xyxy
        pred_classes = result.boxes.cls

        label_path = DETECTION_LABELS_DIR / (img_path.stem + ".txt")
        if not label_path.exists() or len(pred_boxes) == 0:
            continue

        img_w, img_h = result.orig_shape[1], result.orig_shape[0]
        gt_boxes = []
        gt_classes = []
        for line in label_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            parts = line.split()
            cls_id = int(parts[0])
            cx, cy, bw, bh = (float(x) for x in parts[1:5])
            gt_boxes.append([
                (cx - bw / 2) * img_w, (cy - bh / 2) * img_h,
                (cx + bw / 2) * img_w, (cy + bh / 2) * img_h,
            ])
            gt_classes.append(cls_id)

        if not gt_boxes:
            continue

        gt_boxes_t = torch.tensor(gt_boxes)
        gt_classes_t = torch.tensor(gt_classes)
        iou_matrix = box_iou(pred_boxes, gt_boxes_t)

        for i in range(len(pred_boxes)):
            same_class = gt_classes_t == pred_classes[i].item()
            if not same_class.any():
                continue
            best_iou = iou_matrix[i][same_class].max().item()
            if best_iou >= IOU_MATCH_THRESHOLD:
                ious.append(best_iou)

    return sum(ious) / len(ious) if ious else 0.0, len(ious)


summary = {}

print("=== Avaliando detector (Fase 2) no conjunto de TESTE ===")
det_model = YOLO(str(DETECTION_WEIGHTS))
det_results = det_model.val(
    data=str(DETECTION_DATA_YAML), split="test",
    project=str(RESULTS_DIR), name="detection_test", exist_ok=True,
)
summary["detection"] = summarize_val_metrics(det_results)

print("\n=== Calculando IoU medio (TP) do detector no conjunto de TESTE ===")
mean_iou, n_matched = compute_mean_iou_detection()
summary["detection"]["mean_iou_tp"] = mean_iou
summary["detection"]["n_matched_boxes"] = n_matched
print(f"IoU medio (predicoes casadas, conf>={CONF_THRESHOLD}): {mean_iou:.4f} (n={n_matched})")

print("\n=== Avaliando segmentador (Fase 3) no conjunto de TESTE ===")
seg_model = YOLO(str(SEGMENTATION_WEIGHTS))
seg_results = seg_model.val(
    data=str(SEGMENTATION_DATA_YAML), split="test",
    project=str(RESULTS_DIR), name="segmentation_test", exist_ok=True,
)
summary["segmentation"] = {
    "box_precision": float(seg_results.box.mp),
    "box_recall": float(seg_results.box.mr),
    "box_mAP50": float(seg_results.box.map50),
    "box_mAP50-95": float(seg_results.box.map),
    "mask_precision": float(seg_results.seg.mp),
    "mask_recall": float(seg_results.seg.mr),
    "mask_mAP50": float(seg_results.seg.map50),
    "mask_mAP50-95": float(seg_results.seg.map),
}

summary_path = RESULTS_DIR / "summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(f"\nResumo salvo em: {summary_path}")
print(json.dumps(summary, indent=2))


## 12. Análise de erros

Seleciona exemplos de falso positivo/negativo do detector no teste, por maior discrepância de contagem por classe (critério objetivo, não escolha visual).

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

DETECTION_WEIGHTS = ROOT / "models" / "detection" / "css_yolov8n_baseline" / "weights" / "best.pt"
IMAGES_DIR = ROOT / "data" / "raw" / "construction-site-safety" / "images"
LABELS_DIR = ROOT / "data" / "raw" / "construction-site-safety" / "labels"
TEST_LIST = ROOT / "data" / "splits" / "construction-site-safety" / "test.txt"
FIGURES_DIR = ROOT / "reports" / "figures"

CLASS_NAMES = [
    "Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest",
    "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle",
]
CONF_THRESHOLD = 0.25
N_EXAMPLES_EACH = 2


def read_gt_classes(label_path):
    if not label_path.exists():
        return Counter()
    counts = Counter()
    for line in label_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            counts[int(line.split()[0])] += 1
    return counts


FIGURES_DIR.mkdir(parents=True, exist_ok=True)
model = YOLO(str(DETECTION_WEIGHTS))

test_names = [n.strip() for n in TEST_LIST.read_text(encoding="utf-8").splitlines() if n.strip()]

fn_candidates = []
fp_candidates = []

for name in test_names:
    img_path = IMAGES_DIR / name
    label_path = LABELS_DIR / (Path(name).stem + ".txt")

    gt_counts = read_gt_classes(label_path)
    result = model.predict(source=str(img_path), conf=CONF_THRESHOLD, verbose=False)[0]
    pred_counts = Counter(int(c) for c in result.boxes.cls.tolist())

    missing = gt_counts - pred_counts
    extra = pred_counts - gt_counts

    if missing:
        fn_candidates.append((sum(missing.values()), name, missing))
    if extra:
        fp_candidates.append((sum(extra.values()), name, extra))

fn_candidates.sort(key=lambda x: -x[0])
fp_candidates.sort(key=lambda x: -x[0])

examples = []
for score, name, missing in fn_candidates[:N_EXAMPLES_EACH]:
    classes_str = ", ".join(f"{CLASS_NAMES[c]} x{n}" for c, n in missing.items())
    examples.append(("Falso Negativo", name, classes_str, score))
for score, name, extra in fp_candidates[:N_EXAMPLES_EACH]:
    classes_str = ", ".join(f"{CLASS_NAMES[c]} x{n}" for c, n in extra.items())
    examples.append(("Falso Positivo", name, classes_str, score))

for idx, (kind, name, classes_str, score) in enumerate(examples, start=1):
    img_path = IMAGES_DIR / name
    pred_img = model.predict(source=str(img_path), conf=CONF_THRESHOLD, verbose=False)[0].plot()

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(Image.open(img_path))
    axes[0].set_title(f"Original ({name})")
    axes[0].axis("off")

    axes[1].imshow(pred_img[:, :, ::-1])
    axes[1].set_title(f"Predição do modelo (conf>={CONF_THRESHOLD})")
    axes[1].axis("off")

    fig.suptitle(f"{kind}: {classes_str}", fontsize=12)
    plt.tight_layout()
    out_path = FIGURES_DIR / f"erro_{idx}_{kind.lower().replace(' ', '_')}.png"
    fig.savefig(out_path, dpi=150)
    plt.show()
    print(f"{kind} salvo: {out_path} — {classes_str}")


## Resultados esperados

Ver `docs/relatorio-tecnico.md` seção 4.2-5 para os números e a análise de erros completa obtidos no desenvolvimento original (mAP@0.5 detector = 0.547, IoU médio = 0.793).